# RQ4 Explainable AI and Trust

**Research question:** How can Grad-CAM explainability reveal whether a waste-classification model focuses on meaningful object regions?

This Kaggle notebook takes raw image-folder data as input and saves publication-ready tables as CSV and figures as PDF under `/kaggle/working/results/`.

In [7]:

# =========================
# COMMON SETUP
# =========================
import os, time, json, math, random, glob, shutil, pathlib, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

OUTPUT_DIR = Path('/kaggle/working/results')
FIG_DIR = OUTPUT_DIR / 'figures_pdf'
TAB_DIR = OUTPUT_DIR / 'tables_csv'
MODEL_DIR = OUTPUT_DIR / 'models'
for d in [FIG_DIR, TAB_DIR, MODEL_DIR]: d.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (160, 160)
BATCH_SIZE = 32
EPOCHS = 3          # Increase to 10-20 for final results
MAX_IMAGES_PER_CLASS = 500  # Set None for full dataset; keep small for quick Kaggle runs
VALID_EXT = ('.jpg','.jpeg','.png','.webp','.bmp')

# Auto-detect dataset directory. Works with Kaggle datasets and uploaded zip-derived folders.
def find_image_root(base='/kaggle/input'):
    candidates=[]
    for root, dirs, files in os.walk(base):
        image_count=sum(1 for f in files if f.lower().endswith(VALID_EXT))
        if image_count>0:
            candidates.append((root, image_count))
    if not candidates:
        raise FileNotFoundError('No image files found under /kaggle/input. Please attach the dataset in Kaggle Notebook > Add Input.')
    # Prefer a root with class subfolders containing images; otherwise parent of deepest image dirs.
    best=max(candidates, key=lambda x: x[1])[0]
    # If best is a class folder, use parent when multiple sibling class folders exist.
    parent=str(Path(best).parent)
    sibling_img_dirs=[]
    for d in os.listdir(parent):
        p=os.path.join(parent,d)
        if os.path.isdir(p):
            n=sum(1 for f in os.listdir(p) if f.lower().endswith(VALID_EXT))
            if n>0: sibling_img_dirs.append(d)
    if len(sibling_img_dirs)>=2:
        return parent
    # Special case DATASET/TRAIN/TEST: use TRAIN as train root when present.
    for root, dirs, files in os.walk(base):
        if 'TRAIN' in dirs:
            return os.path.join(root,'TRAIN')
    return parent

DATA_ROOT = find_image_root('/kaggle/input')
print('Detected image root:', DATA_ROOT)
print('Class folders:', [d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT,d))][:30])

def build_manifest(data_root=DATA_ROOT, max_per_class=MAX_IMAGES_PER_CLASS):
    rows=[]
    for cls in sorted(os.listdir(data_root)):
        cpath=os.path.join(data_root, cls)
        if not os.path.isdir(cpath): continue
        files=[]
        for r,_,fs in os.walk(cpath):
            files += [os.path.join(r,f) for f in fs if f.lower().endswith(VALID_EXT)]
        if not files: continue
        if max_per_class is not None and len(files)>max_per_class:
            files=random.sample(files, max_per_class)
        for f in files:
            rows.append({'image_path':f, 'label':cls})
    df=pd.DataFrame(rows)
    if df.empty: raise ValueError('No labeled images found. Expected class folders containing images.')
    return df

manifest = build_manifest()
manifest.to_csv(TAB_DIR/'dataset_manifest.csv', index=False)
print(manifest['label'].value_counts())
classes = sorted(manifest['label'].unique())
num_classes=len(classes)
label_to_idx={c:i for i,c in enumerate(classes)}

train_df, temp_df = train_test_split(manifest, test_size=0.30, stratify=manifest['label'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED)
for name,df in [('train',train_df),('val',val_df),('test',test_df)]:
    df.to_csv(TAB_DIR/f'{name}_split.csv', index=False)
    print(name, df.shape)

def make_ds(df, shuffle=False, augment=False):
    paths=df['image_path'].values
    labels=np.array([label_to_idx[x] for x in df['label'].values], dtype=np.int32)
    ds=tf.data.Dataset.from_tensor_slices((paths, labels))
    def load_img(path,label):
        img=tf.io.read_file(path)
        img=tf.image.decode_image(img, channels=3, expand_animations=False)
        img=tf.image.resize(img, IMG_SIZE)
        img=tf.cast(img, tf.float32)/255.0
        if augment:
            img=tf.image.random_flip_left_right(img)
            img=tf.image.random_brightness(img, 0.10)
        return img,label
    ds=ds.map(load_img, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle: ds=ds.shuffle(1000, seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds=make_ds(train_df, shuffle=True, augment=True)
val_ds=make_ds(val_df)
test_ds=make_ds(test_df)

def build_model(model_name):
    inputs=layers.Input(shape=IMG_SIZE+(3,))
    if model_name=='MobileNetV2':
        base=tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_tensor=inputs)
    elif model_name=='EfficientNetB0':
        # inputs already scaled 0-1; EfficientNet preprocessing disabled by using rescaling-neutral setup is acceptable for benchmark simplicity
        base=tf.keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_tensor=inputs)
    elif model_name=='ResNet50':
        base=tf.keras.applications.ResNet50(include_top=False, weights='imagenet', input_tensor=inputs)
    else:
        raise ValueError(model_name)
    base.trainable=False
    x=layers.GlobalAveragePooling2D()(base.output)
    x=layers.Dropout(0.25)(x)
    outputs=layers.Dense(num_classes, activation='softmax')(x)
    model=models.Model(inputs, outputs, name=model_name)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def evaluate_model(model, ds, df):
    y_true=[]; y_pred=[]; probs=[]
    for xb,yb in ds:
        p=model.predict(xb, verbose=0)
        probs.append(p); y_true.extend(yb.numpy().tolist()); y_pred.extend(np.argmax(p,axis=1).tolist())
    y_true=np.array(y_true); y_pred=np.array(y_pred); probs=np.vstack(probs)
    report=classification_report(y_true, y_pred, target_names=classes, output_dict=True, zero_division=0)
    rep_df=pd.DataFrame(report).T
    cm=confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    return rep_df, cm, y_true, y_pred, probs

def measure_latency(model, ds, n_batches=10):
    # Warm-up
    for xb,yb in ds.take(1): model.predict(xb, verbose=0)
    total_imgs=0; start=time.time()
    for i,(xb,yb) in enumerate(ds.take(n_batches)):
        model.predict(xb, verbose=0)
        total_imgs += xb.shape[0]
    elapsed=time.time()-start
    return (elapsed/total_imgs)*1000 if total_imgs else np.nan

def save_pdf(fig, filename):
    path=FIG_DIR/filename
    fig.savefig(path, format='pdf', bbox_inches='tight')
    plt.close(fig)
    print('Saved', path)


Detected image root: /kaggle/input/datasets/techsash/waste-classification-data/DATASET/TRAIN
Class folders: ['R', 'O']
label
O    500
R    500
Name: count, dtype: int64
train (700, 2)
val (150, 2)
test (150, 2)


In [8]:

# RQ4: Explainable AI using Grad-CAM
model=build_model('MobileNetV2')
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1)
rep, cm, y_true, y_pred, probs=evaluate_model(model, test_ds, test_df)
rep.to_csv(TAB_DIR/'RQ4_xai_model_performance_table.csv')

# Find last convolutional layer
last_conv=None
for layer in reversed(model.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv=layer.name; break
if last_conv is None:
    # search nested base model
    for layer in reversed(model.layers[1].layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            last_conv=layer.name; break
print('Last conv layer:', last_conv)

def load_single(path):
    img=tf.io.read_file(path); img=tf.image.decode_image(img, channels=3, expand_animations=False)
    img=tf.image.resize(img, IMG_SIZE); return tf.cast(img, tf.float32)/255.0

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    # For application models nested under full model, target layer lookup may need global get_layer
    conv_layer = model.get_layer(last_conv_layer_name) if last_conv_layer_name in [l.name for l in model.layers] else None
    if conv_layer is None:
        # Use base model layer and build grad model from full model input to base layer output and predictions
        base=model.layers[1]
        conv_layer=base.get_layer(last_conv_layer_name)
    grad_model=tf.keras.models.Model([model.inputs], [conv_layer.output, model.output])
    with tf.GradientTape() as tape:
        conv_outputs, predictions=grad_model(img_array)
        if pred_index is None: pred_index=tf.argmax(predictions[0])
        class_channel=predictions[:, pred_index]
    grads=tape.gradient(class_channel, conv_outputs)
    pooled_grads=tf.reduce_mean(grads, axis=(0,1,2))
    conv_outputs=conv_outputs[0]
    heatmap=conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap=tf.squeeze(heatmap)
    heatmap=tf.maximum(heatmap,0)/tf.math.reduce_max(heatmap + 1e-9)
    return heatmap.numpy()

sample=test_df.sample(min(6, len(test_df)), random_state=SEED).reset_index(drop=True)
fig, axes=plt.subplots(len(sample), 2, figsize=(7, 3*len(sample)))
if len(sample)==1: axes=np.array([axes])
xai_rows=[]
for i,row in sample.iterrows():
    img=load_single(row.image_path); arr=np.expand_dims(img.numpy(),0)
    pred=np.argmax(model.predict(arr, verbose=0)[0]); pred_label=classes[pred]
    try:
        heat=make_gradcam_heatmap(arr, model, last_conv)
        axes[i,0].imshow(img); axes[i,0].axis('off'); axes[i,0].set_title(f'True: {row.label}')
        axes[i,1].imshow(img); axes[i,1].imshow(tf.image.resize(heat[...,None], IMG_SIZE).numpy().squeeze(), alpha=0.45); axes[i,1].axis('off'); axes[i,1].set_title(f'Grad-CAM Pred: {pred_label}')
        xai_rows.append({'image_path':row.image_path,'true_label':row.label,'predicted_label':pred_label,'correct':row.label==pred_label})
    except Exception as e:
        axes[i,0].imshow(img); axes[i,0].axis('off'); axes[i,1].axis('off'); axes[i,1].set_title(str(e)[:50])
pd.DataFrame(xai_rows).to_csv(TAB_DIR/'RQ4_gradcam_sample_table.csv', index=False)
fig.suptitle('RQ4: Explainable AI Evidence Through Grad-CAM Attention Maps', y=1.0)
fig.tight_layout(); save_pdf(fig,'RQ4_gradcam_explainability_panel.pdf')
pd.DataFrame(xai_rows)


Epoch 1/3
22/22 ━━━━━━━━━━━━━━━━━━━━ 16s 479ms/step - accuracy: 0.6929 - loss: 0.6430 - val_accuracy: 0.8933 - val_loss: 0.3287
Epoch 2/3
22/22 ━━━━━━━━━━━━━━━━━━━━ 10s 423ms/step - accuracy: 0.8629 - loss: 0.3354 - val_accuracy: 0.9133 - val_loss: 0.2513
Epoch 3/3
22/22 ━━━━━━━━━━━━━━━━━━━━ 10s 426ms/step - accuracy: 0.8957 - loss: 0.2515 - val_accuracy: 0.9200 - val_loss: 0.2305
Last conv layer: Conv_1
Saved /kaggle/working/results/figures_pdf/RQ4_gradcam_explainability_panel.pdf


,image_path,true_label,predicted_label,correct
0,/kaggle/input/datasets/techsash/waste-classifi...,R,R,True
1,/kaggle/input/datasets/techsash/waste-classifi...,O,O,True
2,/kaggle/input/datasets/techsash/waste-classifi...,R,R,True
3,/kaggle/input/datasets/techsash/waste-classifi...,R,R,True
4,/kaggle/input/datasets/techsash/waste-classifi...,R,O,False
5,/kaggle/input/datasets/techsash/waste-classifi...,R,R,True
